# Retail Sales Regression Pipeline (q3_feature_engineering)

## Load Data

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

df = pd.read_csv('q3_retail_promotions.csv')
df.head()


## 1. Date Feature Engineering

In [ ]:

df['transaction_date'] = pd.to_datetime(df['transaction_date'])

df['year'] = df['transaction_date'].dt.year
df['month'] = df['transaction_date'].dt.month
df['day_of_week'] = df['transaction_date'].dt.dayofweek
df['is_month_end'] = (df['transaction_date'].dt.day >= 25).astype(int)

df.head()


## 2. Temporal Train-Test Split

In [ ]:

df = df.sort_values('transaction_date').reset_index(drop=True)

split_idx = int(len(df)*0.8)
train_df = df.iloc[:split_idx].copy()
test_df = df.iloc[split_idx:].copy()

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Train max date:", train_df['transaction_date'].max())
print("Test min date:", test_df['transaction_date'].min())


**Why not random split?** Randomly mixing future records into training data leaks information from the future and gives unrealistically optimistic performance. Time-ordered splits simulate real forecasting conditions where past data predicts unseen future periods.

## 3. Preprocessing Pipeline

In [ ]:

target = 'items_sold'
drop_cols = ['items_sold','transaction_date']

X_train = train_df.drop(columns=drop_cols)
y_train = train_df[target]

X_test = test_df.drop(columns=drop_cols)
y_test = test_df[target]

categorical_cols = ['promotion_type','location_type','store_size']
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])


## 4. Model Training and Evaluation

In [ ]:

models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(random_state=42, n_estimators=200)
}

results = {}

for name, model in models.items():
    pipe = Pipeline([
        ('prep', preprocessor),
        ('model', model)
    ])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    rmse = mean_squared_error(y_test, preds, squared=False)
    mae = mean_absolute_error(y_test, preds)
    results[name] = {'pipeline': pipe, 'preds': preds, 'rmse': rmse, 'mae': mae}
    print(f"{name}")
    print("RMSE:", round(rmse,3))
    print("MAE :", round(mae,3))
    print("-"*30)


In [ ]:

for name, res in results.items():
    preds = res['preds']
    plt.figure(figsize=(6,6))
    plt.scatter(y_test, preds, alpha=0.7)
    mn = min(y_test.min(), preds.min())
    mx = max(y_test.max(), preds.max())
    plt.plot([mn,mx],[mn,mx], linestyle='--')
    plt.xlabel("Actual items_sold")
    plt.ylabel("Predicted items_sold")
    plt.title(f"Parity Plot - {name}")
    plt.show()


In [ ]:

rf_pipe = results['Random Forest']['pipeline']
ohe = rf_pipe.named_steps['prep'].named_transformers_['cat']
cat_features = ohe.get_feature_names_out(categorical_cols)
all_features = list(cat_features) + numeric_cols

importances = rf_pipe.named_steps['model'].feature_importances_
fi = pd.DataFrame({'feature': all_features, 'importance': importances}).sort_values('importance', ascending=False)

print("Top 5 Features:")
fi.head(5)


**Interpretation:** The top-ranked Random Forest features are the strongest nonlinear predictors of sales. Promotion types, competition density, seasonality, and store characteristics often appear among the most influential variables.